# Benchmark: WebDataset-style shards (tar streaming over S3)


In [ ]:
import io
import json
import os
import tarfile
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from torchvision import transforms

try:
    import fsspec
except ImportError as e:
    raise ImportError('Missing dependency: fsspec (pip install fsspec s3fs)') from e

try:
    from PIL import Image
except ImportError as e:
    raise ImportError('Missing dependency: Pillow (pip install pillow)') from e

print('torch:', torch.__version__)


## Configuration

In [ ]:
S3_BUCKET = os.environ.get('S3_BUCKET', '')
S3_PREFIX = os.environ.get('S3_PREFIX', 'Food-11-webdataset')
SPLIT = os.environ.get('FOOD11_SPLIT', 'evaluation')
S3_ENDPOINT_URL = os.environ.get('S3_ENDPOINT_URL', '')

BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
NUM_WORKERS = 4

WARMUP_BATCHES = 10
MEASURE_BATCHES = 50


if not S3_BUCKET:
    raise ValueError('S3_BUCKET env var is required')

print('S3_BUCKET:', S3_BUCKET)
print('S3_PREFIX:', S3_PREFIX)
print('SPLIT:', SPLIT)
print('S3_ENDPOINT_URL:', S3_ENDPOINT_URL if S3_ENDPOINT_URL else '(default)')
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS:', NUM_WORKERS)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)



## Dataset (stream tar shards)

In [ ]:
fs_kwargs = {}
if S3_ENDPOINT_URL:
    fs_kwargs['client_kwargs'] = {'endpoint_url': S3_ENDPOINT_URL}

fs = fsspec.filesystem('s3', **fs_kwargs)

shard_glob = f"{S3_BUCKET}/{S3_PREFIX}/{SPLIT}/*.tar"
shards = fs.glob(shard_glob)
shards = [s for s in shards if not s.endswith('/')]
shards.sort()

if not shards:
    raise FileNotFoundError(f'No shards found at: s3://{shard_glob}')

print('num_shards:', len(shards))
print('first_shard:', shards[0])

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def _parse_label(b):
    if isinstance(b, bytes):
        b = b.decode('utf-8')
    return int(str(b).strip())

class TarShardDataset(IterableDataset):
    def __init__(self, shards, fs_kwargs, transform=None):
        self.shards = shards
        self.fs_kwargs = fs_kwargs
        self.transform = transform
        self._fs = None

    def _get_fs(self):
        if self._fs is None:
            self._fs = fsspec.filesystem('s3', **self.fs_kwargs)
        return self._fs

    def __iter__(self):
        info = get_worker_info()
        if info is None:
            shard_indices = range(len(self.shards))
        else:
            shard_indices = range(info.id, len(self.shards), info.num_workers)

        fs = self._get_fs()

        for i in shard_indices:
            shard_path = self.shards[i]
            with fs.open(shard_path, 'rb') as f:
                tf = tarfile.open(fileobj=f, mode='r|*')
                current = {}
                current_key = None

                for member in tf:
                    if not member.isfile():
                        continue
                    name = member.name
                    if '.' not in name:
                        continue
                    key, ext = name.rsplit('.', 1)

                    if current_key is None:
                        current_key = key
                    if key != current_key:
                        current = {}
                        current_key = key

                    fh = tf.extractfile(member)
                    if fh is None:
                        continue
                    data = fh.read()
                    current[ext] = data

                    if ('jpg' in current) and ('cls' in current):
                        img = Image.open(io.BytesIO(current['jpg'])).convert('RGB')
                        if self.transform is not None:
                            img = self.transform(img)
                        label = _parse_label(current['cls'])
                        yield img, label

                        current = {}
                        current_key = None

dataset = TarShardDataset(shards=shards, fs_kwargs=fs_kwargs, transform=transform)


## DataLoader

In [ ]:
num_workers = NUM_WORKERS
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    drop_last=False,
    prefetch_factor=2,
    persistent_workers=True,
)


## Run benchmark

In [ ]:
it = iter(loader)

for _ in range(WARMUP_BATCHES):
    try:
        _ = next(it)
    except StopIteration:
        break

num_batches = 0
num_items = 0
t_start = time.perf_counter()
for _ in range(MEASURE_BATCHES):
    try:
        x, y = next(it)
    except StopIteration:
        break
    num_batches += 1
    num_items += int(y.shape[0])
t_end = time.perf_counter()

wall_s = t_end - t_start
imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

result = {
    'num_workers': num_workers,
    'batch_size': BATCH_SIZE,
    'measured_batches': num_batches,
    'measured_items': num_items,
    'wall_s': wall_s,
    'imgs_per_s': imgs_per_s,
    'batches_per_s': batches_per_s,
    'avg_batch_s': avg_batch_s,
}

result


## Print results

In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

avg_batch_s = result['avg_batch_s']
avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
print(
    'workers=', result['num_workers'],
    'imgs/s=', f"{result['imgs_per_s']:.2f}",
    'batches/s=', f"{result['batches_per_s']:.2f}",
    'avg_batch_s=', avg_batch_s_str,
)


## Save results

In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'webdataset_{stamp}.json'
payload = {
    'benchmark': 'webdataset',
    'timestamp_utc': stamp,
    's3_bucket': S3_BUCKET,
    's3_prefix': S3_PREFIX,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'result': result,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
